In [21]:
from datasets import load_dataset

dataset = load_dataset("qintongli/GSM-Plus")
print(dataset)
ds = load_dataset("Maxwell-Jia/AIME_2024")
print(ds)

DatasetDict({
    test: Dataset({
        features: ['question', 'solution', 'answer', 'perturbation_type', 'seed_question', 'seed_solution', 'seed_answer'],
        num_rows: 10552
    })
    testmini: Dataset({
        features: ['question', 'solution', 'answer', 'perturbation_type', 'seed_question', 'seed_solution', 'seed_answer'],
        num_rows: 2400
    })
})
DatasetDict({
    train: Dataset({
        features: ['ID', 'Problem', 'Solution', 'Answer'],
        num_rows: 30
    })
})


In [22]:
%pip install transformers datasets peft accelerate scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [10]:
# data_loader.py

from datasets import load_dataset
import json
import re

def extract_answer_gsm(answer_text):
    # GSM format: ".... #### 42"
    match = re.search(r"####\s*(-?\d+)", answer_text)
    return match.group(1) if match else None


def load_gsm_dataset():
    dataset = load_dataset("gsm8k", "main", split="test")

    data = []
    for item in dataset:
        data.append({
            "question": item["question"],
            "answer": extract_answer_gsm(item["answer"])
        })
    return data


def load_aime_dataset():
    dataset = load_dataset("Maxwell-Jia/AIME_2024", split="train")

    data = []
    for item in dataset:
        data.append({
            "question": item["Problem"],
            "answer": str(item["Answer"])
        })
    return data

In [11]:
# model_utils.py

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_model(model_name="Qwen/Qwen3-0.6B"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE)

    return tokenizer, model


def generate_with_entropy(model, tokenizer, prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            return_dict_in_generate=True,
            output_scores=True
        )

    generated_tokens = outputs.sequences[0]
    scores = outputs.scores  # logits per step

    entropies = []

    for step_logits in scores:
        probs = F.softmax(step_logits[0], dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-9)).item()
        entropies.append(entropy)

    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return generated_text, entropies

In [12]:
# prompts.py

def cot_prompt(question):
    return f"""Solve the following problem step by step:

Question: {question}
Answer:"""


def pot_prompt(question):
    return f"""Write a Python program to solve the problem:

Question: {question}
Program:"""

In [13]:
# experiment.py
import re

def extract_final_number(text):
    match = re.findall(r"-?\d+", text)
    return match[-1] if match else None


def run_experiment(dataset, tokenizer, model):
    results = []

    for item in dataset:
        prompt = cot_prompt(item["question"])

        output, entropies = generate_with_entropy(model, tokenizer, prompt)

        pred = extract_final_number(output)
        correct = (pred == item["answer"])

        results.append({
            "question": item["question"],
            "prediction": pred,
            "ground_truth": item["answer"],
            "correct": correct,
            "entropy_mean": sum(entropies)/len(entropies),
            "entropy_max": max(entropies),
            "entropy_sequence": entropies
        })

    return results

In [14]:
# analysis.py
from sklearn.metrics import roc_auc_score

def compute_auroc(results):
    y_true = [0 if r["correct"] else 1 for r in results]  # 1 = failure
    y_scores = [r["entropy_mean"] for r in results]

    return roc_auc_score(y_true, y_scores)

In [17]:
# Load model
tokenizer, model = load_model()

# Load datasets
gsm_data = load_gsm_dataset()
aime_data = load_aime_dataset()

# Subsample for speed (IMPORTANT)
gsm_data = gsm_data[:2]
aime_data = aime_data[:1]

# Run
gsm_results = run_experiment(gsm_data, tokenizer, model)
print(gsm_results)

aime_results = run_experiment(aime_data, tokenizer, model)
print(aime_results)

# Evaluate
print("GSM AUROC:", compute_auroc(gsm_results))
print("AIME AUROC:", compute_auroc(aime_results))

Loading weights: 100%|██████████| 311/311 [00:05<00:00, 55.33it/s, Materializing param=model.norm.weight]                               
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


[{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'prediction': '3', 'ground_truth': '18', 'correct': False, 'entropy_mean': 0.21743304989649914, 'entropy_max': 1.869239330291748, 'entropy_sequence': [0.5922497510910034, 1.0979533195495605, 1.7221052646636963, 1.7685520648956299, 0.3732338547706604, -0.0, 0.6916958689689636, 1.0546681880950928, 0.6137420535087585, -0.0, -0.0, -0.0, -0.0, 1.6032465696334839, 1.030693531036377, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, 0.6839495897293091, 1.1656728982925415, 0.22453337907791138, 0.8798949718475342, -0.0, -0.0, -0.0, 0.32141679525375366, -0.0, 0.6867285966873169, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, 0.5652915835380554, 0.42440253496170044, -0.0, -0.0, 0.6122605800628662, -0.0, -0.0, 0.60262

c:\Users\sunfl\miniconda3\envs\thesis\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
